# NGRAM = TRUE
# MIN_COUNT = 10
# EMBEDDER = all-MiniLM-L6-v2
# new preprocesssing func

In [ ]:
!pip install gensim top2vec


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from top2vec import Top2Vec
import pandas as pd
from bertopic import BERTopic
from bertopic.backend import MultiModalBackend
from bertopic.representation import KeyBERTInspired
import re
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import random
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

In [5]:
df = pd.read_csv(r"C:\Users\Allen\Documents\GitHub\thesis-research\combined_tweets_dataset\sample_v2\vibe_coding_combined_translated.csv")

In [6]:
df = df[["full_text_translated", "image_url"]]

In [7]:
df.head()

,full_text_translated,image_url
0,@karpathy me vibe coding and hope it can just ...,https://pbs.twimg.com/media/Gi0a82HaQAA97Q-.jpg
1,vibes capital meets vibe coding = greatest eco...,NaN
2,vibe coding https://t.co/1hHVMmunrw,https://pbs.twimg.com/media/Gi0f9fAWgAAgox6.jpg
3,can attest. vibe coding is hella fun and borde...,NaN
4,I have built a few app ideas in my spare time ...,NaN


In [8]:
import html
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Pastikan sudah download resource nltk
# import nltk
# nltk.download('punkt')
# nltk.download('stopwords')

stop_words = set(stopwords.words('english')) # Atau 'indonesian' jika data campur

def preprocess_tweet_v2(text):
    # 1. Decode HTML Entities (Penting untuk X/Twitter data)
    # Ini akan merubah &lt; menjadi < sehingga re.sub simbol bisa bekerja
    text = html.unescape(text)

    text = text.lower()

    # 2. Pola Regex kamu sudah bagus, pertahankan untuk URL, Mentions, Hashtags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)

    # 3. Handle spesifik Vibe Coding sebelum hapus simbol
    text = re.sub(r'vibe\s+coding', 'vibecoding', text)
    text = re.sub(r'vibe\s+code', 'vibecode', text)
    text = re.sub(r'vibe\s+coded', 'vibecoded', text)

    # 4. Hapus simbol dan angka
    text = re.sub(r'[^\w\s]', ' ', text) # Ganti simbol dengan spasi agar kata tidak menempel
    text = re.sub(r'\d+', '', text)

    # 5. Tokenisasi & Stopwords (The missing piece)
    tokens = word_tokenize(text)
    cleaned_tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    return " ".join(cleaned_tokens)

In [9]:
df_preprocessed = df['full_text_translated'].fillna('').apply(preprocess_tweet_v2)
df_preprocessed = pd.concat([df_preprocessed, df['image_url']], axis=1)

In [10]:
df_preprocessed = df_preprocessed[df_preprocessed['full_text_translated'].apply(lambda x: len(x.split()) >= 4)]
df_preprocessed.count()

full_text_translated    19049
image_url                6000
dtype: int64

In [11]:
docs = df_preprocessed['full_text_translated'].tolist()
images  = df_preprocessed['image_url'].tolist()
for i in range(len(images)):
    if pd.isna(images[i]):
        images[i] = None
random.shuffle(docs)
docs[:10]

['vibecoding making realize much time wasted memorizing syntax senior dev less knowing language taste tell vibes change mind',
 'vibecoding build sooo easy game everyone also vibecoding spending hrs get layout would taken mins figma',
 'hey hear give mobile app connects ide allows guide junie could vibecoding many part already place',
 'vibecoding ignites urge create hour launched website personal projects angel investing holding forever',
 'far best article read today art vibecoding',
 'vibecoding vibe everything knowledge work change way coding changed past year imagine labs let anthropic lead long complex multi step projects computer access become standard',
 'know customize theme knack app blends seamlessly brand experience',
 'insane vibecoding without coding',
 'vibecoding hand tacking robotics mafia',
 'crazy much cripple trying vibecode selecting wrong tech stack right coding models gods react vite make separate backend simple hono workers next use next']

In [13]:
# Top2Vec topic modeling
documents = [doc for doc in docs if isinstance(doc, str) and doc.strip()]
tokenized_documents = [doc.split() for doc in documents]

dictionary = Dictionary(tokenized_documents)
dictionary.filter_extremes(no_below=2, no_above=0.5)

print(f"tokenized documents: {tokenized_documents[:5]}")
print(f"Number of documents: {len(documents)}")
print(f"Number of unique tokens: {len(dictionary)}")



tokenized documents: [['vibecoding', 'making', 'realize', 'much', 'time', 'wasted', 'memorizing', 'syntax', 'senior', 'dev', 'less', 'knowing', 'language', 'taste', 'tell', 'vibes', 'change', 'mind'], ['vibecoding', 'build', 'sooo', 'easy', 'game', 'everyone', 'also', 'vibecoding', 'spending', 'hrs', 'get', 'layout', 'would', 'taken', 'mins', 'figma'], ['hey', 'hear', 'give', 'mobile', 'app', 'connects', 'ide', 'allows', 'guide', 'junie', 'could', 'vibecoding', 'many', 'part', 'already', 'place'], ['vibecoding', 'ignites', 'urge', 'create', 'hour', 'launched', 'website', 'personal', 'projects', 'angel', 'investing', 'holding', 'forever'], ['far', 'best', 'article', 'read', 'today', 'art', 'vibecoding']]
Number of documents: 19049
Number of unique tokens: 10517


In [14]:
print(dictionary.token2id)

{'change': 0, 'dev': 1, 'knowing': 2, 'language': 3, 'less': 4, 'making': 5, 'memorizing': 6, 'mind': 7, 'much': 8, 'realize': 9, 'senior': 10, 'syntax': 11, 'taste': 12, 'tell': 13, 'time': 14, 'vibes': 15, 'wasted': 16, 'also': 17, 'build': 18, 'easy': 19, 'everyone': 20, 'figma': 21, 'game': 22, 'get': 23, 'hrs': 24, 'layout': 25, 'mins': 26, 'sooo': 27, 'spending': 28, 'taken': 29, 'would': 30, 'allows': 31, 'already': 32, 'app': 33, 'connects': 34, 'could': 35, 'give': 36, 'guide': 37, 'hear': 38, 'hey': 39, 'ide': 40, 'junie': 41, 'many': 42, 'mobile': 43, 'part': 44, 'place': 45, 'angel': 46, 'create': 47, 'forever': 48, 'holding': 49, 'hour': 50, 'investing': 51, 'launched': 52, 'personal': 53, 'projects': 54, 'urge': 55, 'website': 56, 'art': 57, 'article': 58, 'best': 59, 'far': 60, 'read': 61, 'today': 62, 'access': 63, 'anthropic': 64, 'become': 65, 'changed': 66, 'coding': 67, 'complex': 68, 'computer': 69, 'everything': 70, 'imagine': 71, 'knowledge': 72, 'labs': 73, 'lea

In [15]:
import top2vec
print(top2vec.__version__)

1.0.36


In [16]:
top2vec_model = Top2Vec(
    documents=documents,
    ngram_vocab=True,
    min_count=10,
    embedding_model='all-MiniLM-L6-v2',    
    speed='learn',
    workers=2,
    keep_documents=True,
    verbose=True,
 )

topic_words, topic_word_scores, topic_nums = top2vec_model.get_topics()



2026-04-12 20:38:03,613 - top2vec - INFO - Pre-processing documents for training
2026-04-12 20:38:04,712 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2000.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-12 20:38:11,123 - top2vec - INFO - Creating joint document/word embedding
2026-04-12 20:38:19,712 - top2vec - INFO - Creating lower dimension embedding of documents
2026-04-12 20:38:39,917 - top2vec - INFO - Finding dense areas of documents
2026-04-12 20:38:43,795 - top2vec - INFO - Finding topics


In [17]:
topic_sizes, topic_ids = top2vec_model.get_topic_sizes()
pd.DataFrame({
    "topic_id": topic_ids,
    "size": topic_sizes
}).sort_values("size", ascending=False).head(10)

,topic_id,size
0,0,1615
1,1,1004
2,2,975
3,3,739
4,4,706
5,5,656
6,6,654
7,7,619
8,8,595
9,9,559


In [18]:
print(len(topic_sizes))

81


In [19]:
# Coherence metrics for Top2Vec topics (C_V and NPMI)
topic_pairs = []
for topic_num, topic in zip(topic_nums, topic_words):
    cleaned_topic_terms = []
    for term in topic[:10]:
        # ngram terms like "vibe coding" are split so they can be matched to dictionary tokens
        for tok in str(term).split():
            if tok in dictionary.token2id:
                cleaned_topic_terms.append(tok)
    # keep token order, drop duplicates
    cleaned_topic_terms = list(dict.fromkeys(cleaned_topic_terms))
    if len(cleaned_topic_terms) >= 2:
        topic_pairs.append((topic_num, cleaned_topic_terms))

topics_for_coherence = [words for _, words in topic_pairs]
topic_nums_for_coherence = [topic_num for topic_num, _ in topic_pairs]

if not topics_for_coherence:
    raise ValueError("No valid topics left for coherence calculation. Try lowering dictionary filtering or increasing topic words.")

coherence_cv_model = CoherenceModel(
    topics=topics_for_coherence,
    texts=tokenized_documents,
    dictionary=dictionary,
    coherence='c_v',
)

coherence_npmi_model = CoherenceModel(
    topics=topics_for_coherence,
    texts=tokenized_documents,
    dictionary=dictionary,
    coherence='c_npmi',
)

coherence_cv = coherence_cv_model.get_coherence()
coherence_npmi = coherence_npmi_model.get_coherence()

print(f"Total topics used for coherence: {len(topics_for_coherence)}")
print(f"C_V coherence: {coherence_cv:.4f}")
print(f"NPMI coherence: {coherence_npmi:.4f}")

topic_coherence = pd.DataFrame({
    'topic_num': topic_nums_for_coherence,
    'topic_words': [' '.join(words) for words in topics_for_coherence],
    'c_v': coherence_cv_model.get_coherence_per_topic(),
    'npmi': coherence_npmi_model.get_coherence_per_topic(),
})

topic_coherence.sort_values('c_v', ascending=False).head(10)

Total topics used for coherence: 81
C_V coherence: 0.2972
NPMI coherence: -0.1572


,topic_num,topic_words,c_v,npmi
46,46,ideas reality mission critical idea plain rage...,0.417706,-0.252046
61,61,vibe debugging vibecoded projects firebase stu...,0.356801,-0.220805
78,78,rick rubin vibe marketing investing designing ...,0.350044,-0.218114
52,52,vibecoded projects vibe coded gpt codex dont v...,0.339827,-0.177153
73,73,claude sonnet vibe coding vibecoded projects c...,0.331902,-0.163621
9,9,agentforce vibes vibecoded projects vibe coded...,0.321060,-0.115827
51,51,trading bot vibe investing marketing coded don...,0.318701,-0.169322
59,59,vibe investing vibecoded projects coded market...,0.317502,-0.155279
5,5,vibe coded vibecode camp vibecoded projects de...,0.317502,-0.155279
70,70,vibe investing marketing coding coded vibecode...,0.317502,-0.155279


In [20]:
topic_words_values = topic_coherence.sort_values('c_v', ascending=False).head(10)['topic_words'].values
print(topic_words_values)

['ideas reality mission critical idea plain rage coding check content creators bring boring stuff instead writing elon musk'
 'vibe debugging vibecoded projects firebase studio vibecode camp designing cleanup coded marketing dont investing'
 'rick rubin vibe marketing investing designing coding vibecoded projects vibecode camp coded wordpress dont'
 'vibecoded projects vibe coded gpt codex dont vibecode debugging camp chat coding cleanup investing'
 'claude sonnet vibe coding vibecoded projects coded dont vibecode designing camp debugging investing agentforce vibes'
 'agentforce vibes vibecoded projects vibe coded coding designing debugging marketing investing agentic engineering vibecode camp'
 'trading bot vibe investing marketing coded dont vibecode telegram coding agentforce vibes vibecoded projects debugging'
 'vibe investing vibecoded projects coded marketing vibecode camp coding agentforce vibes designing debugging dont'
 'vibe coded vibecode camp vibecoded projects designing co

In [ ]:
# # Save model
# top2vec_model.save("top2vec_ngram_c10.model")

# # Load model (di masa depan)
# # from top2vec import Top2Vec
# top2vec_model = Top2Vec.load("top2vec_ngram_c10.model")



# # Load di kemudian hari
# # embeddings = np.load("top2vec_ngram_c10_embeddings.npy")


# # Simpan Dictionary
# dictionary.save("top2vec_ngram_c10.dict")

# # Simpan tokenized_documents (menggunakan pickle karena list of lists)
# import pickle
# with open("top2vec_ngram_c10_tokenized_docs.pkl", "wb") as f:
#     pickle.dump(tokenized_documents, f)
    
    
    
# # Simpan ringkasan topik dan metriknya
# topic_coherence.to_csv("top2vec_ngram_c10_metrics_summary.csv", index=False)

# # Simpan representasi topik (10 kata teratas)
# top_words_df = pd.DataFrame(topic_words)
# top_words_df.to_csv("top2vec_ngram_c10_topic_words.csv")